# Análise de experimentos UAV V2

Lê os eventos de `/v2/status` gravados no Postgres e gera gráficos de:
- **taxa de sucesso** por experimento
- **distribuição de passos** até concluir

Pré-requisito: banco de experimentos rodando (ver `README.md`) e as libs:
```bash
pip install pandas matplotlib seaborn "psycopg[binary]" jupyter
```

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg

sns.set_theme(style="whitegrid")
%matplotlib inline


In [ ]:
DB_URL = "postgresql://v2:v2@localhost:5432/v2_experiments"

conn = psycopg.connect(DB_URL)
events = pd.read_sql(
    "SELECT id, experiment_id, ts, event_type, payload "
    "FROM experiment_events ORDER BY experiment_id, ts, id",
    conn,
)
print(f"{len(events)} eventos carregados")
events.head()

In [ ]:
# Constrói missões: agrupa eventos de started..finished/error
missions = []
cur = []
for _, ev in events.iterrows():
    t = ev["event_type"]
    if t == "started":
        cur = [ev]
    elif t in ("finished", "error") and cur:
        cur.append(ev)
        missions.append(cur)
        cur = []
    elif cur:
        cur.append(ev)
print(f"{len(missions)} missões identificadas")

In [ ]:
rows = []
for m in missions:
    exp = m[0]["experiment_id"]
    steps = [e for e in m if e["event_type"] == "vlm_step"]
    n_steps = int(steps[-1]["payload"]["step"]) if steps else 0
    finished = [e for e in m if e["event_type"] == "finished"]
    success = bool(finished and str(finished[0]["payload"].get("success", "false")).lower() == "true")
    duration_s = (finished[0]["ts"] - m[0]["ts"]).total_seconds() if finished else None
    rows.append({"experiment": exp, "success": success, "steps": n_steps, "duration_s": duration_s})
df = pd.DataFrame(rows)
df.head()

In [ ]:
# Taxa de sucesso por experimento
taxa = df.groupby("experiment")["success"].mean().sort_values()
ax = taxa.plot(kind="barh", figsize=(8, 4), color="#4C72B0")
ax.set_xlabel("Taxa de sucesso")
ax.set_title("Taxa de sucesso por experimento")
for i, v in enumerate(taxa):
    ax.text(v + 0.01, i, f"{v:.1%}", va="center")
plt.tight_layout()
plt.savefig("outputs/taxa_sucesso.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Distribuição de passos até concluir (missões com sucesso)
ok = df[df["success"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(ok["steps"], bins=10, ax=ax[0], color="#55A868")
ax[0].set_title("Histograma de passos (sucesso)")
sns.boxplot(data=ok, x="experiment", y="steps", ax=ax[1], palette="Set2")
ax[1].set_title("Boxplot de passos por experimento")
ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("outputs/passos_missao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Tabela resumo (pronta para o artigo, via to_latex)
resumo = df.groupby("experiment").agg(
    missoes=("success", "count"),
    sucesso=("success", "sum"),
    taxa=("success", "mean"),
    passos_mediana=("steps", "median"),
    passos_std=("steps", "std"),
    duracao_mediana_s=("duration_s", "median"),
).round(2)
resumo
# Para LaTeX: print(resumo.to_latex())

## Exportar para o artigo

Os PNGs ficam em `experiments/outputs/`. Para usar no artigo IEEE:

```latex
\begin{figure}[t]
  \centering
  \includegraphics[width=\columnwidth]{../npl-drone-tracking/experiments/outputs/taxa_sucesso.png}
  \caption{Taxa de sucesso por experimento.}
  \label{fig:taxa_sucesso}
\end{figure}
```

Os números da tabela resumo alimentam o `% TODO(evidência)` do abstract.